# 📊 Media Literacy App - Action Logs Analysis Notebook
**Dataset**: `docs/analytics/action_logs.csv` (Exported from Supabase PostgreSQL)
**Period**: July 19 - August 12, 2026
**Total Records**: 16,525 Events | **Unique Sessions**: 123 Players

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

plt.style.use('seaborn-v0_8-whitegrid')
print('Setup Completed')

In [ ]:
# Load Action Logs Dataset
df = pd.read_csv('action_logs.csv')
df['created_at'] = pd.to_datetime(df['created_at'])
df['date'] = df['created_at'].dt.date

def parse_payload(val):
    if pd.isna(val) or not isinstance(val, str):
        return {}
    try:
        return json.loads(val)
    except:
        return {}

df['payload_dict'] = df['payload'].apply(parse_payload)
print(f'Loaded {len(df)} total logs across {df["session_id"].nunique()} unique sessions.')

In [ ]:
# Key Metrics Summary
print('--- Overview Summary ---')
print('Total Events:', len(df))
print('Unique Players:', df['session_id'].nunique())
print('Date Range:', df['created_at'].min(), 'to', df['created_at'].max())

# Date Breakdown
date_summary = df.groupby('date').agg(total_actions=('id', 'count'), unique_players=('session_id', 'nunique')).reset_index()
print('\n--- Daily Activity ---')
print(date_summary)

In [ ]:
# Top 15 Events
top_events = df.groupby('event_name').agg(total_count=('id', 'count'), unique_users=('session_id', 'nunique')).reset_index().sort_values(by='total_count', ascending=False)
print(top_events.head(15))

In [ ]:
# Plot Daily Event Timeline
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot([d.strftime('%m-%d') for d in date_summary['date']], date_summary['total_actions'], marker='o', color='#8B5CF6')
ax.set_title('Daily Event Timeline')
ax.set_ylabel('Total Actions')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()